# Logistic regression

In [ ]:
!pip install -q transformers sentence-transformers accelerate bitsandbytes

In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, accuracy_score,
    precision_score, recall_score, f1_score, roc_curve
)
from scipy.stats import pearsonr

from huggingface_hub import login
import random

# 使用 HuggingFace Token
HF_TOKEN = "hf_XXXXXXXXXXXXXXXXXXXXX" 
login(HF_TOKEN)

# 硬體設定
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 隨機變數
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# BATCH_SIZE
BATCH_SIZE = 1 # RAGTruth

In [ ]:
def load_multi_json(path):
    decoder = json.JSONDecoder()
    objs = []

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    idx = 0
    n = len(text)

    while idx < n:
        # skip whitespace
        while idx < n and text[idx].isspace():
            idx += 1
        if idx >= n:
            break

        obj, end = decoder.raw_decode(text, idx)
        objs.append(obj)
        idx = end

    return objs

In [ ]:
# ====================================================
# 1. DataFrame 建立
# ====================================================
def construct_dataframe(result_file):

    print(f"[Info] 讀取結果檔案: {result_file}")
    if not os.path.exists(result_file):
        raise FileNotFoundError(f"找不到檔案: {result_file}")

    records = []
    records = load_multi_json(result_file)
    
    if not records:
        raise ValueError("結果檔案中沒有任何有效資料")

    first_score = records[0]["scores"][0]
    ecs_keys = list(first_score["prompt_attention_score"].keys())
    pks_keys = list(first_score["parameter_knowledge_scores"].keys())

    data = {
        "identifier": [],
        "hallucination_label": [],
        "split": [],
    }

    for k in ecs_keys:
        data[f"ecs_{k}"] = []
    for k in pks_keys:
        data[f"pks_{k}"] = []

    for ridx, item in enumerate(tqdm(records, desc="Building DataFrame")):
        split = item["split"]
        for sidx, score in enumerate(item["scores"]):
            data["identifier"].append(f"resp_{ridx}_span_{sidx}")
            data["hallucination_label"].append(score["hallucination_label"])
            data["split"].append(split)

            for k in ecs_keys:
                data[f"ecs_{k}"].append(score["prompt_attention_score"].get(k, 0.0))
            for k in pks_keys:
                data[f"pks_{k}"].append(score["parameter_knowledge_scores"].get(k, 0.0))

    df = pd.DataFrame(data)
    print("\n[Label Distribution]")
    print(df["hallucination_label"].value_counts(normalize=True))

    return df, ecs_keys, pks_keys

In [ ]:
# ====================================================
# 2. Response leakage check
# ====================================================
def check_response_leakage(df):
    df = df.copy()
    df["resp"] = df["identifier"].apply(lambda x: x.split("_span_")[0])
    bad = df.groupby("resp")["split"].nunique()
    bad = bad[bad > 1]
    if len(bad) > 0:
        raise RuntimeError("Response leakage detected")
    print("No response leakage detected")

In [ ]:
# ====================================================
# 3. Feature AUC ranking
# ====================================================
def rank_features(df, keys, prefix):
    y = df["hallucination_label"].values
    scored = []
    for k in keys:
        col = f"{prefix}_{k}"
        try:
            auc = roc_auc_score(y, df[col].values)
        except:
            auc = 0.5
        scored.append((k, auc))
    return [k for k, _ in sorted(scored, key=lambda x: x[1], reverse=True)]

In [ ]:
# ====================================================
# 4. ReDeEP Regression + Threshold Learning
# ====================================================
def evaluate_redeep(train_df, test_df, ecs_keys, pks_keys):

    train_df = train_df.copy()
    test_df = test_df.copy()

    train_df["ecs_sum"] = train_df[[f"ecs_{k}" for k in ecs_keys]].sum(axis=1)
    train_df["pks_sum"] = train_df[[f"pks_{k}" for k in pks_keys]].sum(axis=1)
    test_df["ecs_sum"] = test_df[[f"ecs_{k}" for k in ecs_keys]].sum(axis=1)
    test_df["pks_sum"] = test_df[[f"pks_{k}" for k in pks_keys]].sum(axis=1)

    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(train_df[["pks_sum", "ecs_sum"]])
    X_test = scaler.transform(test_df[["pks_sum", "ecs_sum"]])

    y_train = train_df["hallucination_label"].values
    y_test = test_df["hallucination_label"].values

    model = LogisticRegression(solver="liblinear", random_state=RANDOM_SEED)
    model.fit(X_train, y_train)

    train_score = model.decision_function(X_train)
    test_score = model.decision_function(X_test)

    fpr, tpr, th = roc_curve(y_train, train_score)
    threshold = th[np.argmax(tpr - fpr)]

    pred_chunk = (test_score > threshold).astype(int)

    auc_c = roc_auc_score(y_test, test_score)
    pcc_c = pearsonr(test_score, y_test)[0]
    acc_c = accuracy_score(y_test, pred_chunk)
    prec_c = precision_score(y_test, pred_chunk, zero_division=0)
    rec_c = recall_score(y_test, pred_chunk, zero_division=0)
    f1_c = f1_score(y_test, pred_chunk, zero_division=0)

    test_df["score"] = test_score
    test_df["pred"] = pred_chunk
    test_df["resp"] = test_df["identifier"].apply(lambda x: x.split("_span_")[0])

    grp = test_df.groupby("resp").agg({
        "score": "mean",
        "hallucination_label": "max",
        "pred": "max"
    })

    auc_r = roc_auc_score(grp["hallucination_label"], grp["score"])
    pcc_r = pearsonr(grp["score"], grp["hallucination_label"])[0]
    acc_r = accuracy_score(grp["hallucination_label"], grp["pred"])
    prec_r = precision_score(grp["hallucination_label"], grp["pred"], zero_division=0)
    rec_r = recall_score(grp["hallucination_label"], grp["pred"], zero_division=0)
    f1_r = f1_score(grp["hallucination_label"], grp["pred"], zero_division=0)

    return {
        "auc_chunk": auc_c, "acc_chunk": acc_c,
        "precision_chunk": prec_c, "recall_chunk": rec_c,
        "f1_chunk": f1_c, "pcc_chunk": pcc_c,
        "auc_resp": auc_r, "acc_resp": acc_r,
        "precision_resp": prec_r, "recall_resp": rec_r,
        "f1_resp": f1_r, "pcc_resp": pcc_r,
        "alpha": model.coef_[0,0],
        "beta": -model.coef_[0,1],
        "threshold": threshold,
        "all_ecs_train": train_df["ecs_sum"],
        "all_pks_train": train_df["pks_sum"],
        "all_ecs_test": test_df["ecs_sum"],
        "all_pks_test": test_df["pks_sum"]
    }

In [ ]:
# ====================================================
# 5. Main
# ====================================================
if __name__ == "__main__":

    MODEL_KEY = "Llama-3.2-1B"
    
    # ===================== 基本設定 =====================
    INPUT_RESULT_FILE = "Llama-3.2-1B_ragtruth_results.json"   # JSON 輸入 根據 stage 2 的輸出改名字
    SAVE_PATH = f"redeep_eval_result_{MODEL_KEY}.json"                      # 儲存最終結果 (JSON)
    

    df, ecs_keys, pks_keys = construct_dataframe(INPUT_RESULT_FILE)
    check_response_leakage(df)

    train_df = df[df["split"] == "train"]
    test_df  = df[df["split"] == "test"]

    ecs_ranked = rank_features(train_df, ecs_keys, "ecs")
    pks_ranked = rank_features(train_df, pks_keys, "pks")

    best = None
    best_params = None

    if MODEL_KEY == "Llama-3.2-1B":
        cand_ecs = list(range(1, 17))   # [1, 2, ..., 16]
        cand_pks = list(range(1, 17))   # [1, 2, ..., 16]
    elif MODEL_KEY == "Llama-3.2-3B":
        cand_ecs = list(range(1, 29))   # [1, 2, ..., 16]
        cand_pks = list(range(1, 29))   # [1, 2, ..., 16]


    for n_ecs in cand_ecs:
        for n_pks in cand_pks:

            n_ecs_eff = min(n_ecs, len(ecs_ranked))
            n_pks_eff = min(n_pks, len(pks_ranked))

            sel_ecs_keys = ecs_ranked[:n_ecs_eff]
            sel_pks_keys = pks_ranked[:n_pks_eff]

            res = evaluate_redeep(
                train_df=train_df,
                test_df=test_df,
                ecs_keys=sel_ecs_keys,
                pks_keys=sel_pks_keys
            )

            if (best is None) or (res["auc_resp"] > best["auc_resp"]):
                best = res
                best_params = {"top_ecs": n_ecs, "top_pks": n_pks,"ecs_keys": list(sel_ecs_keys), "pks_keys": list(sel_pks_keys)}

    # ---- 6. 印出最終結果 ----
    print("\n================ 最終結果 (ReDeEP Regression) ================")
    print(f"最佳 Top-ECS-head 數量: {best_params['top_ecs']} → {best_params['ecs_keys']}")
    print(f"最佳 Top-PKS-layer 數量: {best_params['top_pks']} → {best_params['pks_keys']}")
    print(f"學到的 PKS 係數 (alpha): {best['alpha']:.4f}")
    print(f"學到的 ECS 係數 (beta):  {best['beta']:.4f}")
    print(f"學到的 threshold:  {best['threshold']:.4f}")

    print("\n[Chunk-level]")
    print(f"  AUC:      {best['auc_chunk']:.4f}")
    print(f"  PCC:      {best['pcc_chunk']:.4f}")
    print(f"  Accuracy: {best['acc_chunk']:.4f}")
    print(f"  Precision:{best['precision_chunk']:.4f}")
    print(f"  Recall:   {best['recall_chunk']:.4f}")
    print(f"  F1:       {best['f1_chunk']:.4f}")

    print("\n[Response-level]")
    print(f"  AUC:      {best['auc_resp']:.4f}")
    print(f"  PCC:      {best['pcc_resp']:.4f}")
    print(f"  Accuracy: {best['acc_resp']:.4f}")
    print(f"  Precision:{best['precision_resp']:.4f}")
    print(f"  Recall:   {best['recall_resp']:.4f}")
    print(f"  F1:       {best['f1_resp']:.4f}")

# NNLS

In [ ]:
import os
import json
import numpy as np
import pandas as pd

from tqdm import tqdm
from scipy.optimize import nnls
from scipy.stats import pearsonr

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
def load_multi_json(path):
    """
    Load a file containing multiple JSON objects concatenated together.
    """
    decoder = json.JSONDecoder()
    objs = []

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    idx = 0
    n = len(text)

    while idx < n:
        # skip whitespace
        while idx < n and text[idx].isspace():
            idx += 1
        if idx >= n:
            break

        obj, end = decoder.raw_decode(text, idx)
        objs.append(obj)
        idx = end

    return objs

In [ ]:
# ====================================================
# 0. DataFrame 建構（適用 HalluRAG / 你現在的格式）
# ====================================================

def construct_dataframe(result_file):
    print(f"[Info] 讀取結果檔案: {result_file}")
    if not os.path.exists(result_file):
        raise FileNotFoundError(f"找不到檔案: {result_file}")

    records = []
    records = load_multi_json(result_file)
    
    if not records:
        raise ValueError("結果檔案中沒有任何有效資料")

    # 取第一個 span 看有哪些 ECS / PKS key
    first_score = records[0]["scores"][0]
    ecs_keys = list(first_score["prompt_attention_score"].keys())
    pks_keys = list(first_score["parameter_knowledge_scores"].keys())

    print(f"[Info] 偵測到 ECS heads 數量 = {len(ecs_keys)}, PKS layers 數量 = {len(pks_keys)}")

    data = {
        "identifier": [],
        "hallucination_label": [],
        "split": [],
    }
    for k in ecs_keys:
        data[f"ecs_{k}"] = []
    for k in pks_keys:
        data[f"pks_{k}"] = []

    for ridx, item in enumerate(tqdm(records, desc="建構 DataFrame")):
        split = item["split"]
        for sidx, score in enumerate(item["scores"]):
            data["identifier"].append(f"resp_{ridx}_span_{sidx}")
            data["hallucination_label"].append(score["hallucination_label"])
            data["split"].append(split)

            for k in ecs_keys:
                data[f"ecs_{k}"].append(
                    score["prompt_attention_score"].get(k, 0.0)
                )
            for k in pks_keys:
                data[f"pks_{k}"].append(
                    score["parameter_knowledge_scores"].get(k, 0.0)
                )

    df = pd.DataFrame(data)

    print("\n[Info] 標籤分布 (0: 非幻覺, 1: 幻覺)")
    print(df["hallucination_label"].value_counts(normalize=True))

    return df, ecs_keys, pks_keys


# ====================================================
# Feature ranking
# ====================================================

def calculate_feature_metrics(df, feature_keys, prefix, label_col="hallucination_label"):
    metrics = []
    y = df[label_col].values

    for key in feature_keys:
        col = f"{prefix}_{key}"
        x = df[col].values
        try:
            auc = roc_auc_score(y, x)
        except Exception:
            auc = 0.5
        metrics.append({"key": key, "auc": auc})

    return metrics


# ====================================================
# Utils
# ====================================================

def safe_pearsonr(x, y):
    try:
        return pearsonr(x, y)[0]
    except Exception:
        return None

# ====================================================
# ReDeEP NNLS
# ====================================================

def train_redeep_nnls(train_df, pks_keys, ecs_keys):
    pks_cols = [f"pks_{k}" for k in pks_keys]
    ecs_cols = [f"ecs_{k}" for k in ecs_keys]
    feature_cols = pks_cols + ecs_cols

    X_raw = train_df[feature_cols].values.astype(float)
    y = train_df["hallucination_label"].values.astype(float)

    scaler = MinMaxScaler()
    X = scaler.fit_transform(X_raw)

    coef, _ = nnls(X, y)

    alpha = coef[:len(pks_cols)]   # PKS 部分
    beta = coef[len(pks_cols):]    # ECS 部分

    return alpha, beta, scaler, feature_cols


def apply_redeep(df, alpha, beta, scaler, feature_cols, pks_keys):
    X = scaler.transform(df[feature_cols].values.astype(float))
    n_pks = len(pks_keys)

    score = X[:, :n_pks] @ alpha - X[:, n_pks:] @ beta

    df = df.copy()
    df["redeep_score"] = score
    return df

In [ ]:
# ====================================================
# Evaluation
# ====================================================

def evaluate_with_threshold(test_df, threshold):
    # ---------- Chunk-level ----------
    y_c = test_df["hallucination_label"].values
    s_c = test_df["redeep_score"].values
    yhat_c = (s_c >= threshold).astype(int)

    chunk = {
        "AUC": roc_auc_score(y_c, s_c) if len(np.unique(y_c)) > 1 else None,
        "Accuracy": accuracy_score(y_c, yhat_c),
        "Precision": precision_score(y_c, yhat_c, zero_division=0),
        "Recall": recall_score(y_c, yhat_c, zero_division=0),
        "F1": f1_score(y_c, yhat_c, zero_division=0),
        "PCC": safe_pearsonr(s_c, y_c),
    }

    # ---------- Response-level ----------
    df = test_df.copy()
    df["pred"] = yhat_c
    df["response_group"] = df["identifier"].apply(lambda x: x.split("_span_")[0])

    grouped = df.groupby("response_group").agg({
        "redeep_score": "mean",
        "hallucination_label": "max",
        "pred": "max"
    }).reset_index()

    y_r = grouped["hallucination_label"].values
    s_r = grouped["redeep_score"].values
    yhat_r = grouped["pred"].values

    response = {
        "AUC": roc_auc_score(y_r, s_r) if len(np.unique(y_r)) > 1 else None,
        "Accuracy": accuracy_score(y_r, yhat_r),
        "Precision": precision_score(y_r, yhat_r, zero_division=0),
        "Recall": recall_score(y_r, yhat_r, zero_division=0),
        "F1": f1_score(y_r, yhat_r, zero_division=0),
        "PCC": safe_pearsonr(s_r, y_r),
    }

    return chunk, response

In [ ]:
# ====================================================
# Main
# ====================================================

INPUT_RESULT_FILE = "Llama-3.2-1B_ragtruth_results.json"   # JSON 輸入 根據 stage 2 的輸出改名字
SAVE_PATH = "redeep_nnls_eval.json"

if __name__ == "__main__":

    # ---------- Load data ----------
    df, ecs_keys, pks_keys = construct_dataframe(INPUT_RESULT_FILE)

    # 使用 dataset 內建 split
    train_df = df[df["split"] == "train"].copy()
    test_df  = df[df["split"] == "test"].copy()

    # safety：檢查 response leakage
    assert len(
        set(train_df["identifier"].str.split("_span_").str[0])
        & set(test_df["identifier"].str.split("_span_").str[0])
    ) == 0, "Response leakage detected!"

    # ---------- Feature ranking ----------
    ecs_metrics = calculate_feature_metrics(train_df, ecs_keys, "ecs")
    pks_metrics = calculate_feature_metrics(train_df, pks_keys, "pks")

    ecs_ranked = [m["key"] for m in sorted(ecs_metrics, key=lambda x: x["auc"], reverse=True)]
    pks_ranked = [m["key"] for m in sorted(pks_metrics, key=lambda x: x["auc"], reverse=True)]

    # ---------- Grid search ----------
    best_auc = -1
    best_all = None

    total_ecs = len(ecs_ranked)
    total_pks = len(pks_ranked)

    print("\n[Info] 開始 Grid Search (Top-ECS x Top-PKS)...")
    for n_ecs in range(1, total_ecs + 1):
        for n_pks in range(1, total_pks + 1):

            sel_ecs = ecs_ranked[:n_ecs]
            sel_pks = pks_ranked[:n_pks]

            alpha, beta, scaler, feature_cols = train_redeep_nnls(
                train_df, sel_pks, sel_ecs
            )

            train_scored = apply_redeep(
                train_df, alpha, beta, scaler, feature_cols, sel_pks
            )

            threshold = 0.0
            
            fpr, tpr, thresholds = roc_curve(train_scored["hallucination_label"].values, train_scored["redeep_score"].values)
            j_scores = tpr - fpr
            best_idx = np.argmax(j_scores)
            threshold = thresholds[best_idx]

            test_scored = apply_redeep(
                test_df, alpha, beta, scaler, feature_cols, sel_pks
            )

            chunk, response = evaluate_with_threshold(test_scored, threshold)

            # # 迴圈進度
            # print(
            #     f"n_ecs={n_ecs:2d}, n_pks={n_pks:2d} "
            #     f"-> AUC_resp={response['AUC']:.4f}, AUC_chunk={chunk['AUC']:.4f}, alpha={alpha.tolist()}, beta: {beta.tolist()}"
            # )

            if response["AUC"] is not None and response["AUC"] > best_auc:
                best_auc = response["AUC"]
                best_all = {
                    "top_ecs": sel_ecs,
                    "top_pks": sel_pks,
                    "alpha": alpha.tolist(),
                    "beta": beta.tolist(),
                    "threshold": float(threshold),
                    "chunk": chunk,
                    "response": response
                }

    print("\n================ 最終結果 (ReDeEP NNLS) ================")
    print(f"最佳 Top-ECS-head 數量: {len(best_all['top_ecs'])} → {best_all['top_ecs']}")
    print(f"最佳 Top-PKS-layer 數量: {len(best_all['top_pks'])} → {best_all['top_pks']}")
    print(f"學到的 PKS 係數 (alpha, mean): {np.mean(best_all['alpha']):.4f}")
    print(f"學到的 ECS 係數 (beta, mean):  {np.mean(best_all['beta']):.4f}")
    print(f"學到的 threshold: {best_all['threshold']:.4f}")

    table = pd.DataFrame([
        {"Level": "Chunk", **best_all["chunk"]},
        {"Level": "Response", **best_all["response"]},
    ])

    print("\n================ Evaluation Metrics ================")
    print(table.round(4))